# 01. Naive RAG의 한계

## 실습 개요

이번 실습에서는 **DoclingLoader로 PDF를 Markdown 형태로 로딩한 뒤**,  
LangChain의 `RecursiveCharacterTextSplitter`를 사용해 단순한 크기 기반 Chunking을 수행한다.

이 방식은 Docling의 문서 파싱 능력은 활용하지만, Docling의 구조 기반 Chunking은 사용하지 않는다.  
따라서 이후 실습에서 `DOC_CHUNKS + HybridChunker` 방식과 비교하기 좋은 **Naive RAG 기준선(baseline)** 이 된다.

```text
PDF
 ↓
DoclingLoader
 ↓
ExportType.MARKDOWN
 ↓
LangChain Document
 ↓
RecursiveCharacterTextSplitter
 ↓
Embedding
 ↓
ChromaDB
 ↓
Retriever
 ↓
LLM
```

## Naive RAG의 주요 한계

| 문제 | 원인 | 영향 |
|---|---|---|
| 검색 품질 불안정 | 단일 쿼리에만 의존 | 질문 표현에 따라 검색 결과 편차 발생 |
| 노이즈 컨텍스트 | 관련도가 낮아도 k개 문서를 반환 | LLM 답변 품질 저하 가능 |
| 컨텍스트 중복 | 유사한 청크가 반복 검색 | 토큰 낭비 |
| 단순 Chunking | 문서 구조보다 길이 중심으로 분할 | 의미 단위가 끊길 수 있음 |
| 검색 실패 | 필요한 근거 문서를 찾지 못함 | 부정확하거나 근거가 약한 답변 가능 |

## 학습 목표

실습 후 다음 내용을 설명할 수 있다.

1. `DoclingLoader`를 사용해 PDF를 LangChain `Document`로 로딩할 수 있다.
2. `ExportType.MARKDOWN`의 역할을 설명할 수 있다.
3. `RecursiveCharacterTextSplitter`를 이용한 Naive Chunking을 구현할 수 있다.
4. 검색 결과와 distance score를 확인할 수 있다.
5. Naive RAG의 검색·청킹 한계를 설명할 수 있다.

## Docling 이해

Docling은 PDF, DOCX, PPTX 같은 문서를 단순히 문자열로 추출하는 것을 넘어  
제목, 본문, 표, 목록, 문서 계층 등 **문서 구조를 분석할 수 있는 문서 처리 라이브러리**다.

이번 1번 실습에서는 Docling이 분석한 문서를 **Markdown으로 export**한 뒤  
LangChain의 일반적인 Text Splitter로 청킹한다.

> 핵심: **1번은 MARKDOWN + 일반 청킹**,  
> 이후 2번은 **DOC_CHUNKS + Docling 구조 기반 청킹**으로 비교한다.


## DoclingLoader 사용 방법

LangChain 기반 RAG에서는 Docling의 `DocumentConverter`를 직접 다루는 대신  
`langchain-docling`의 `DoclingLoader`를 사용하면 문서 로딩 흐름을 단순화할 수 있다.

### 설치

```shell
uv add langchain-docling
```

### 기본 사용

```python
from langchain_docling import DoclingLoader

loader = DoclingLoader(file_path="sample.pdf")
documents = loader.load()
```

`DoclingLoader`의 기본 export 방식은 `DOC_CHUNKS`이다.  
이번 실습에서는 Naive Chunking을 직접 수행하기 위해 **`ExportType.MARKDOWN`을 명시적으로 사용한다.**


In [16]:
# 추가 설치가 필요한 패키지
# DoclingLoader는 langchain-docling 패키지에서 제공된다.
#
# uv add langchain-docling chromadb langchain-chroma langchain-text-splitters langchain-openai ipywidgets


## 실습 데이터
**「2026 AI 주요 트렌드 전망」NIA AI 트렌드 보고서 기반 한국어 RAG 챗봇 만들기** 
- data/AI@Data_Report_2026_전망_251223(최종).pdf

# 환경변수 로딩

In [17]:
import sys
sys.path.append("..")
from common_config import embedding_model, llm_connect

## PDF 문서 로딩: DoclingLoader + MARKDOWN

이번 실습에서는 PDF를 `DoclingLoader`로 읽고 `MARKDOWN` 형태의 LangChain `Document`로 변환한다.

- Docling: PDF의 레이아웃과 문서 구조를 파싱
- `ExportType.MARKDOWN`: 파싱 결과를 Markdown 문서로 반환
- Chunking: 아직 수행하지 않음
- 다음 단계에서 `RecursiveCharacterTextSplitter`로 단순 청킹

따라서 이 단계에서는 **문서 로딩과 Chunking의 역할을 분리**해서 이해한다.


In [18]:
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from docling.datamodel.pipeline_options import PdfPipelineOptions

pipeline_options = PdfPipelineOptions()

# 실습에 사용할 PDF 파일
PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")


# ---------------------------------------------------------
# 1. Docling의 PDF 파싱 옵션 설정
# ---------------------------------------------------------
# OCR이 필요하지 않은 텍스트 기반 PDF이므로 OCR은 비활성화한다.
# 스캔 PDF라면 do_ocr=True가 필요할 수 있다.
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False
pipeline_options.allow_external_plugins = True


# ---------------------------------------------------------
# 2. DocumentConverter 구성
# ---------------------------------------------------------
# DoclingLoader 내부에서 사용할 PDF 변환기를 만든다.
# 이후 2번 실습에서도 동일한 파싱 조건을 사용하면
# MARKDOWN 방식과 DOC_CHUNKS 방식의 차이를 공정하게 비교할 수 있다.
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


# ---------------------------------------------------------
# 3. DoclingLoader로 PDF 로딩
# ---------------------------------------------------------
# MARKDOWN 모드는 문서를 Markdown 형태의 LangChain Document로 반환한다.
# 여기서는 Docling의 native chunking을 사용하지 않는다.
loader = DoclingLoader(
    file_path=str(PDF_PATH),
    converter=converter,
    export_type=ExportType.MARKDOWN,
)

documents = loader.load()

print(f"로딩된 LangChain Document 수: {len(documents)}")


Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

로딩된 LangChain Document 수: 1


In [19]:
documents

[Document(metadata={'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf'}, page_content="## AI@Data Report\n\n## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망\n\n2025-제3호\n\n## 목차\n\n- ❶ 배경 및 목적 /p.06\n- ❷ 자료 분석 및 방법 /p.09\n- ❸ 분석 결과 /p.12\n- ❹ 2026년 전망: AI 생태계 방향성 /p.17\n- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20\n\n## AI@Data Report\n\n2025 제 3 호\n\n(2025. 12월)\n\n## 토픽 분석을 통한\n\n## AI 주요 트렌드 및 2026 전망\n\nAI@Data 보고서 시리즈는...\n\nAI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.\n\n- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이므로, 본 보고서의 내용을 발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다.\n- ❷ 보고서 내용의 무단전재를 금하며, 가공‧인용할 때는 반드시 출처를 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다.\n- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 공식 견해와 다를 수 있습니다.\n\n작성 한국지능정보사회진흥원 김시연 수석(siyeon@nia.or.kr, 053-230-4249)\n\n추지혜 책임(chuu@nia.or.kr, 053-230-4247)\n\n전민석 주임(msj1224@nia.or.kr, 053-230-4205)\n\n## 기획 한국지능정보사회진흥원 인공지능데이터본부\n\n신신애 본부장(sashin@nia.or.kr, 053-230-4200)\n\n심호찬 팀장(s

In [20]:
# DoclingLoader가 반환한 첫 번째 Document 확인
# 현재 PDF는 1개이므로 일반적으로 documents[0]에 전체 Markdown 문서가 들어간다.

doc = documents[0]

print("[metadata]")
print(doc.metadata)

print("\n[Markdown 내용 일부]")
print(doc.page_content[:1500])


[metadata]
{'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf'}

[Markdown 내용 일부]
## AI@Data Report

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

2025-제3호

## 목차

- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

## AI@Data Report

2025 제 3 호

(2025. 12월)

## 토픽 분석을 통한

## AI 주요 트렌드 및 2026 전망

AI@Data 보고서 시리즈는...

AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.

- ❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의 '초거대 AI 데이터 발굴 및 기획' 사업의 산출물이므로, 본 보고서의 내용을 발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다.
- ❷ 보고서 내용의 무단전재를 금하며, 가공‧인용할 때는 반드시 출처를 「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다.
- ❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의 공식 견해와 다를 수 있습니다.

작성 한국지능정보사회진흥원 김시연 수석(siyeon@nia.or.kr, 053-230-4249)

추지혜 책임(chuu@nia.or.kr, 053-230-4247)

전민석 주임(msj1224@nia.or.kr, 053-230-4205)

## 기획 한국지능정보사회진흥원 인공지능데이터본부

신신애 본부장(sashin@nia.or.kr, 053-230-4200)

심호찬 팀장(shc@nia.or.kr, 053-230-4201)

## 표·그림 목차

| [표1]

In [21]:
# Markdown 문서의 전체 길이 확인
# 아직 Chunking 전이므로 하나의 큰 문서 상태이다.

markdown_text = documents[0].page_content

print(f"전체 문자 수: {len(markdown_text):,}")
print(f"전체 줄 수: {len(markdown_text.splitlines()):,}")


전체 문자 수: 21,788
전체 줄 수: 411


In [22]:
from pathlib import Path

# Docling이 변환한 Markdown 결과를 직접 확인하고 싶을 때 저장한다.
# RAG 실행 자체에 반드시 필요한 단계는 아니며, 문서 파싱 결과 점검용이다.

OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_docling.md"

MD_PATH.write_text(
    documents[0].page_content,
    encoding="utf-8",
)

print("Markdown 저장 완료:")
print(MD_PATH.resolve())


Markdown 저장 완료:
/home/kiho/rag_agent_2/3.advanced_rag_ai_trend/data/AI@Data_Report_2026_전망_251223(최종)_docling.md


## Naive Chunking: RecursiveCharacterTextSplitter

Docling은 구조 기반 Chunking도 지원하지만, **이번 실습에서는 의도적으로 사용하지 않는다.**

`MARKDOWN`으로 로딩한 문서를 `RecursiveCharacterTextSplitter`로 다시 나누어  
일반적인 Naive RAG의 Chunking 방식을 구현한다.

```text
DoclingLoader
    ↓
MARKDOWN Document
    ↓
RecursiveCharacterTextSplitter
    ↓
500자 기준 Chunk
```

이 방식은 단순하고 이해하기 쉽지만 다음과 같은 한계가 있다.

- 제목과 본문이 서로 다른 청크로 분리될 수 있다.
- 표가 중간에서 나뉠 수 있다.
- 의미 단위보다 `chunk_size`에 더 크게 의존한다.
- 문서 구조를 충분히 활용하지 못한다.

이 한계는 2번 실습의 `DOC_CHUNKS + HybridChunker`와 비교한다.


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ---------------------------------------------------------
# 1. Naive Chunking 설정
# ---------------------------------------------------------
# chunk_size:
#   하나의 청크에 포함할 최대 문자 수의 기준값
#
# chunk_overlap:
#   인접 청크 사이에 일부 내용을 중복시켜
#   경계에서 문맥이 완전히 끊기는 것을 완화한다.
#
# 주의:
#   이 방식은 Docling의 구조 기반 Chunking이 아니라
#   LangChain의 일반적인 문자열 기반 Chunking이다.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n", ".", " ", ""],
)


# ---------------------------------------------------------
# 2. LangChain Document를 여러 Chunk로 분할
# ---------------------------------------------------------
chunks = splitter.split_documents(documents)


# ---------------------------------------------------------
# 3. 검색 결과 추적을 위해 chunk_id 추가
# ---------------------------------------------------------
# splitter는 기존 metadata를 각 chunk에 복사한다.
# 여기에 실습에서 검색 결과를 확인하기 쉽도록 chunk_id만 추가한다.
for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = i


print(f"분할된 청크 수: {len(chunks)}")
print("청크 크기 기준: 500")
print("청크 중복: 80")

print("\n[첫 번째 청크 예시]")
print("metadata:", chunks[0].metadata)
print(chunks[0].page_content[:500])


분할된 청크 수: 73
청크 크기 기준: 500
청크 중복: 80

[첫 번째 청크 예시]
metadata: {'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 1}
## AI@Data Report

## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

2025-제3호

## 목차

- ❶ 배경 및 목적 /p.06
- ❷ 자료 분석 및 방법 /p.09
- ❸ 분석 결과 /p.12
- ❹ 2026년 전망: AI 생태계 방향성 /p.17
- ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20

## AI@Data Report

2025 제 3 호

(2025. 12월)

## 토픽 분석을 통한

## AI 주요 트렌드 및 2026 전망

AI@Data 보고서 시리즈는...

AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.


# ChromaDB로 Vector DB 구성

Markdown 문서를 Naive 방식으로 청킹한 `chunks`를 임베딩하여 ChromaDB에 저장한다.

> Chunking 전략을 변경했을 때 이전 Vector DB를 그대로 재사용하면 실험 결과가 섞일 수 있다.  
> 따라서 이번 실습에서는 MARKDOWN 기반 Naive RAG 전용 DB 경로를 사용한다.


## ChromaDB 생성 또는 로딩

In [24]:
from pathlib import Path

from common_config import embedding_model
from langchain_chroma import Chroma


# 임베딩 모델 초기화
embeddings = embedding_model()


# ---------------------------------------------------------
# MARKDOWN + RecursiveCharacterTextSplitter 전용 Vector DB
# ---------------------------------------------------------
# 기존 text 기반 DB와 섞이지 않도록 별도 경로/컬렉션명을 사용한다.
DB_PATH = Path("chroma_db/ai_trend_report_naive_markdown")
COLLECTION_NAME = "ai_trend_report_2026_naive_markdown"


# 기존 DB가 있으면 그대로 로딩한다.
if DB_PATH.exists():
    print("기존 ChromaDB가 존재하므로 새로 생성하지 않고 로딩합니다.")

    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(DB_PATH),
    )

# 기존 DB가 없으면 현재 chunks로 새로 생성한다.
else:
    print("기존 ChromaDB가 없어 새로 생성합니다.")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(DB_PATH),
        # cosine distance를 명시적으로 사용하려면 아래 옵션을 활성화할 수 있다.
        # collection_metadata={"hnsw:space": "cosine"},
    )


print(f"\n저장된 청크 수: {vector_store._collection.count()}")


기존 ChromaDB가 존재하므로 새로 생성하지 않고 로딩합니다.

저장된 청크 수: 73


### ChromaDB 기본 유사도 계산
| 항목       | Chroma DB 기본값            |
| -------- | ------------------------ |
| 기본 거리 방식 | `l2` (제곱 유클리드 거리)  |
| 의미       | Squared L2 distance      |
| 해석       | 낮을수록 더 유사                |
| 변경 가능 값  | `cosine`, `ip`           |
| 설정 위치    | collection 생성 시 metadata |

### ChromaDB의 score 해석

`similarity_search_with_score()`에서 반환되는 score는 일반적인 “유사도 점수”라기보다 거리값에 가깝다.

따라서 기본 설정에서는 다음과 같이 해석한다.

- score가 낮을수록 질문과 문서가 더 가깝다.
- score가 높을수록 질문과 문서의 거리가 멀다.
- ChromaDB의 기본 거리 방식은 L2 distance 계열이다.
  - 유클리드 거리 제곱 = Squared L2 Distance, 벡터 간 거리의 제곱. 기본값
- cosine 기반 거리를 사용하려면 collection 생성 시 별도로 지정해야 한다.

예시:

```python
Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="ai_trend_report_2026_pdf_cosine",
    persist_directory="chroma_db/ai_trend_report_pdf_cosine",
    collection_metadata={"hnsw:space": "cosine"}
)
```

# 벡터DB 쿼리 및 LLM 답변 결과 확인

## Naive RAG 한계 확인

In [25]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# LLM 객체 생성(gpt-5.4-mini)
llm = llm_connect()

# top-k 검색을 위한 retriever 생성
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 검색된 문서들을 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", """아래 문서를 참고해서 질문에 답하세요. 문서에 없는 내용은 모른다고 답변하세요.
     [문서] {context}     
     """),
    ("human", "{question}")
])

# LCEL 체인 정의
naive_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [26]:
retriever

VectorStoreRetriever(tags=['Chroma', 'OpenAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x77bbbcde92b0>, search_kwargs={'k': 3})

In [27]:
# [질문]
# 1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
# 2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
# 3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
# 4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
# 5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?

In [28]:
# Docling MARKDOWN 기반 VectorDB의 검색 결과 확인
question = "2026년 AI 기술 전망에서 추론형 AI는 어떻게 발전하나요?"

#results = naive_chain.invoke(question)
results = retriever.invoke(question)

print("\n[검색 결과]")
#print(f"검색된 문서 수: {len(results)}")
for i, doc in enumerate(results[:2], start=1):
    print(f"[{i}] metadata={doc.metadata}")
    print(doc.page_content[:300])
    print("---")



[검색 결과]
[1] metadata={'chunk_id': 9, 'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf'}
## ■ 2026년 AI 생태계 방향성 전망

- 산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있음
- Ÿ 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로 예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨

## 분석 배경 및 목적 CHAPTER1

## 1.1 분석 배경
---
[2] metadata={'source': 'data/AI@Data_Report_2026_전망_251223(최종).pdf', 'chunk_id': 55}
## AI 산업·기술·정책의 2026년 전망 CHAPTER4

## 4.1 2025년 AI 산업·기술·정책의 트렌드 현황

- 앞 장에서는 2025년 주요 매체 데이터를 바탕으로 산업·기술·정책 전반에서 나타난 AI 담론의
- 구조적 흐름을 도출하였음
- Ÿ 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화 요구 증가 → 제도적 대응 심화로 이어지는 연속적 흐름을 형성하고 있음을 확인하였음
- 2026년에는 이 세 방향성이 더욱 맞물려 산업 구조의 재배치, 기술 체계의 정교화,
- 규제·책임 
---


In [29]:
# 사용자의 질문에 대해 Vector Store가 어떤 청크를 검색했는지 확인하는 검색 결과 점검 함수
def inspect_retrieval(query: str, k: int = 5, active_store=vector_store):
    """
    질문에 대해 검색된 문서를 확인한다.

    similarity_search_with_score()의 score는 기본적으로 distance 값이다.
    즉, 낮을수록 질문과 문서가 더 가깝다.
    """

    results = active_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수 k] {k}")
    print("ChromaDB distance score: 낮을수록 더 유사함")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page", None)
        page_label = page + 1 if isinstance(page, int) else "?"
        chunk_id = doc.metadata.get("chunk_id", "?")
        source = doc.metadata.get("source", "?")

        content = doc.page_content[:500].replace("\n", " ")

        print(f"\n[{i}] score={score:.4f} | page={page_label} | chunk_id={chunk_id}")
        print(f"source={source}")
        print(content)

In [30]:
# 문제 1: 표현 방식에 따른 검색 편차 확인

test_queries = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?",
]

for query in test_queries:
    inspect_retrieval(query, k=3)
    print("\n")

[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.6762 | page=? | chunk_id=9
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## ■ 2026년 AI 생태계 방향성 전망  - 산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI 생태계의 구조 재편이 가속화될 가능성이 있음 - Ÿ 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로 예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨  ## 분석 배경 및 목적 CHAPTER1  ## 1.1 분석 배경

[2] score=0.7887 | page=? | chunk_id=8
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## ■ 2025년 AI 산업·기술·정책 주요 담론  - (도입 확산과 가치사슬 변화) 기업과 산업 영역에서 AI의 적용 범위가 넓어지며, 시장의 성장 속도·도입 깊이·운영 체계 변화가 주요 쟁점으로 부상함 - (지능 구조 고도화와 적용 환경 다변화) 멀티모달·추론·경량화 등 내부 알고리즘이 고도화 되고, 기술 적용이 클라우드에서 온디바이스 등으로 확대되는 양상이 나타남 - (안전·규범 중심의 제도화 심화) 고위험군 평가, 책임 기준 명료화, 데이터·저작권 쟁점 정비 등 위험 관리 중심의 제도 설계가 빠르게 진전되고, 국제 규제와의 정합성을 맞추기 위한 대응도 확대되는 흐름이 확인됨  ## ■ 2026년 AI 생태계 방향성 전망

[3] score=0.7912 | page=? | chunk_id=55
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## AI 산업·기술·정책의 2026년 전망 CHAPTER4

## 실제 Naive RAG 답변 비교

In [31]:
def ask_naive_rag(question: str):
    """
    Naive RAG 체인을 실행하고 답변을 출력한다.
    """
    answer = naive_chain.invoke(question)

    print("=" * 100)
    print(f"[질문]\n{question}")
    print("\n[Naive RAG 답변]")
    print(answer)

    return answer

In [32]:
rag_questions = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
]

for question in rag_questions:
    ask_naive_rag(question)

[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[Naive RAG 답변]
문서에 따르면 2026년 AI **기술 전망의 핵심 변화**는 다음으로 정리됩니다.

- **기능 고도화가 더 심화됨**
  - 멀티모달, 추론, 경량화 같은 **내부 알고리즘의 고도화**가 계속 진전될 것으로 봅니다.

- **적용 환경이 더 다양해짐**
  - 기술 적용이 **클라우드 중심에서 온디바이스 등으로 확대**되는 양상이 나타날 것으로 전망됩니다.

즉, 2026년 AI 기술은 **더 똑똑해지고(고도화), 더 넓은 환경에서 쓰이는 방향(적용 환경 다변화)**으로 변화할 것으로 문서는 설명합니다.
[질문]
내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?

[Naive RAG 답변]
문서 기준으로 보면, 내년 AI 모델은 **지능 구조 고도화와 데이터 한계 보완**을 중심으로 발전할 것으로 보입니다. 핵심 방향은 다음과 같습니다.

- **합성데이터 활용 확대**
  - 고품질 데이터를 보완하기 위해 합성데이터가 중요한 축이 될 것으로 전망됩니다.

- **추론형 AI 강화**
  - 단순 생성보다 복잡한 상황에서의 판단, 문제 해결 능력이 더 중요해질 것으로 보입니다.

- **멀티모달 기술 고도화**
  - 텍스트뿐 아니라 이미지, 음성 등 다양한 입력을 함께 처리하는 능력이 강화될 전망입니다.

- **학습 효율 및 설명 가능성 개선**
  - 모델 내부 구조의 질적 개선을 통해 더 효율적으로 학습하고, 결과를 더 잘 설명하는 방향으로 발전할 것으로 예상됩니다.

- **활용 환경 확대**
  - 클라우드 중심을 넘어 다양한 적용 환경에서 활용될 가능성이 커지고, 서비스·산업 전반의 적용도 넓어질 것으로 보입니다.

또한 이런 기술 고도화가 진행될수록 **안전성, 정합성 검증, 신뢰성 평가, 표준화**의 중요성도 함께 커질 것으로 문서는 전망하고 있습니다.

문서에 없는 세부 모델명이나 구체적 성능 수치는 알 수 없습니다.
[질문]
산업·기술·정

## 검색 문서와 답변 함께 확인하기

In [33]:
def compare_retrieval_and_answer(question: str, k: int = 3):
    """
    검색 결과와 최종 답변을 함께 확인한다.
    Naive RAG에서 검색 품질이 답변 품질에 어떤 영향을 주는지 확인할 수 있다.
    """

    print("\n\n")
    print("#" * 100)
    print("[1] 검색 결과 확인")
    print("#" * 100)
    inspect_retrieval(question, k=k)

    print("\n\n")
    print("#" * 100)
    print("[2] Naive RAG 답변 확인")
    print("#" * 100)
    answer = naive_chain.invoke(question)
    print(answer)

    return answer

In [34]:
compare_retrieval_and_answer(
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=3
)




####################################################################################################
[1] 검색 결과 확인
####################################################################################################
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.6793 | page=? | chunk_id=27
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
&lt; 표 2 &gt; 분야별 데이터 수집 키워드  | 분야   | 사용 키워드                                                             | |--------|-------------------------------------------------------------------------| | AI산업 | AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환          | | AI기술 | AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발     | | AI정책 | AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 |  [ 그림 1 ] 분야별 데이터셋 구성 예시  ## 2.2 연구 방법

[2] score=0.8140 | page=? | chunk_id=25
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
- Ÿ 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 - 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심

'문서의 <표 2>에 따르면 분야별 데이터 수집 키워드는 다음과 같습니다.\n\n- **AI산업**: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환\n- **AI기술**: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발\n- **AI정책**: AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책'

## 관련 없는 청크도 반환

### 확인 포인트

Naive RAG는 `k=5`로 설정하면 관련성이 낮은 문서가 포함되더라도 무조건 5개의 청크를 반환한다.

이 방식의 문제는 다음과 같다.

1. 관련성이 낮은 문서가 프롬프트에 함께 들어간다.
2. LLM이 중요하지 않은 내용을 함께 참고할 수 있다.
3. 답변이 길어지거나 초점이 흐려질 수 있다.
4. 토큰 사용량이 증가한다.

이 문제는 이후 실습에서 `Reranking`과 `Contextual Compression`으로 개선한다.

In [35]:
### 관련 없는 청크도 반환
# [질문]
# 1. AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
# 2. 보고서에서 말하는 온디바이스 AI 확산은 어떤 의미인가요?
# 3. 토픽모델링에서 단순 출현 빈도보다 중요하게 본 기준은 무엇인가요?
# 4. AI 정책에서 출력물 표시와 데이터 출처 공개는 왜 중요하게 다루어지나요?
# 5. 보고서의 전처리 과정에서 제거한 단어 유형은 무엇인가요?

In [36]:
# 문제 2: 관련 없는 청크도 반환
query =  "AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?"
inspect_retrieval(query, k=5)

[질문] AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.8733 | page=? | chunk_id=57
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
- 데이터·AI 중심 구조로 이동하는 압력이 강화되는 양상을 보일 것으로 전망됨 - 기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며 - 사람-에이전트-시스템이 혼합된 업무 구조가 일부 영역에서 확산될 가능성이 있음 - Ÿ 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할 것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는 수준으로 평가됨

[2] score=0.8950 | page=? | chunk_id=42
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
- 인프라, 센터, 효율, 에이전트: 인프라·에이전트 기반 운영 모델 변화 - Ÿ 데이터센터·클라우드 기반의 인프라 투자 확대 경향을 반영함 - Ÿ 에이전트 도입 확산은 업무 흐름과 운영 방식이 재구성되는 초기 징후로 볼 수 있음 - 현시점 AI 산업은 시장 규모 확대, 도입 가속, 인프라 중심 운영 모델화가 동시에 나타나며, AI가 산업의 핵심 인프라로 자리 잡아가는 흐름을 시사함

[3] score=0.9388 | page=? | chunk_id=51
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
|---------|-------------------------------------|----------------------------------------------------------------------------------------------|-----------

## 표 기반 검색 한계

In [37]:
### 문제3. 표 기반 검색 한계
# [질문]
# 1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
# 2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
# 3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
# 4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
# 5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [38]:
query =  "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 과정를 거쳤나요?"
inspect_retrieval(query, k=5)

[질문] 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 과정를 거쳤나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.8408 | page=? | chunk_id=6
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 한눈에 보기 PREVIEW  ## ■ 분석 배경 및 목적  - 본 이슈페이퍼는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로, 산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 간략히 조망하는 것을 목적으로 함 - 산업·기술·정책 분야별 텍스트에 LDA 토픽모델링 기법을 적용하여, 각각의 영역에서 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료로 활용하고자 함  ## ■ 분석 절차

[2] score=0.8815 | page=? | chunk_id=1
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## AI@Data Report  ## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망  2025-제3호  ## 목차  - ❶ 배경 및 목적 /p.06 - ❷ 자료 분석 및 방법 /p.09 - ❸ 분석 결과 /p.12 - ❹ 2026년 전망: AI 생태계 방향성 /p.17 - ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20  ## AI@Data Report  2025 제 3 호  (2025. 12월)  ## 토픽 분석을 통한  ## AI 주요 트렌드 및 2026 전망  AI@Data 보고서 시리즈는...  AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해 미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해 한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.

[3] score=0.9771 | page=?

In [39]:
query =  "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
inspect_retrieval(query, k=5)

[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.6793 | page=? | chunk_id=27
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
&lt; 표 2 &gt; 분야별 데이터 수집 키워드  | 분야   | 사용 키워드                                                             | |--------|-------------------------------------------------------------------------| | AI산업 | AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환          | | AI기술 | AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발     | | AI정책 | AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 |  [ 그림 1 ] 분야별 데이터셋 구성 예시  ## 2.2 연구 방법

[2] score=0.8140 | page=? | chunk_id=25
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
- Ÿ 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 - 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중 산업·기술·정책의 주요 이슈와 변화 방향을 가장 정확히 대표하는 단어들을 선별하여 구성 - Ÿ 핵심 키워드는 산업·기술·정책 각 분야에서 반복적으로 등장한 상위 빈도 핵심어 중 대표성이 가장 높은 단어를 기반으로 선정함 - Ÿ 이는 각 키워드가 분야별 담론 구조와 주요 이슈·변화 방향(시장 확산, 기술 고도화, 규제 강화 등)을 가장 명확하게 설명하는 지표 역할을 하기 때문임 - 

In [40]:
rag_questions = [
    "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
]

for question in rag_questions:
    compare_retrieval_and_answer(question)




####################################################################################################
[1] 검색 결과 확인
####################################################################################################
[질문] 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.8357 | page=? | chunk_id=6
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망 한눈에 보기 PREVIEW  ## ■ 분석 배경 및 목적  - 본 이슈페이퍼는 2025년 국내외 주요 매체에서 수집한 AI 관련 텍스트를 바탕으로, 산업·기술·정책 세 영역에서 형성되는 담론 구조와 주요 흐름을 간략히 조망하는 것을 목적으로 함 - 산업·기술·정책 분야별 텍스트에 LDA 토픽모델링 기법을 적용하여, 각각의 영역에서 반복적으로 등장하는 핵심 논점과 의미 축을 도출함으로써 2026년 AI 생태계를 전망하기 위한 기초 자료로 활용하고자 함  ## ■ 분석 절차

[2] score=0.8806 | page=? | chunk_id=1
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
## AI@Data Report  ## 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망  2025-제3호  ## 목차  - ❶ 배경 및 목적 /p.06 - ❷ 자료 분석 및 방법 /p.09 - ❸ 분석 결과 /p.12 - ❹ 2026년 전망: AI 생태계 방향성 /p.17 - ❺ 분석 결과로 본 우리의 대응 및 노력 /p.20  ## AI@Data Report  2025

In [41]:
result = compare_retrieval_and_answer(
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
    k=5
)




####################################################################################################
[1] 검색 결과 확인
####################################################################################################
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] score=0.6793 | page=? | chunk_id=27
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
&lt; 표 2 &gt; 분야별 데이터 수집 키워드  | 분야   | 사용 키워드                                                             | |--------|-------------------------------------------------------------------------| | AI산업 | AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환          | | AI기술 | AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발     | | AI정책 | AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책 |  [ 그림 1 ] 분야별 데이터셋 구성 예시  ## 2.2 연구 방법

[2] score=0.8140 | page=? | chunk_id=25
source=data/AI@Data_Report_2026_전망_251223(최종).pdf
- Ÿ 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함 - 본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심

In [42]:
print(result)

문서에 따르면 분야별 데이터 수집 키워드는 다음과 같습니다.

- 산업(AI산업): AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환
- 기술(AI기술): AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발
- 정책(AI정책): AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책
